# Figure 12 -- forward vs forward+backward cost

Loads `results/differentiability/autodiff_overhead.json`, produced by `bench/differentiability/autodiff_overhead.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.repo_root() / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("differentiability/autodiff_overhead.json")
cfg, recs = art["config"], art["data"]["records"]
recs = [r for r in recs if "error" not in r]
if not recs:
    raise SystemExit("autodiff_overhead.json has no successful rows")

bases = [b for b in cfg["basis"] if any(r["basis"] == b for r in recs)]
lanes = sorted({r["lane"] for r in recs})
wrts = list(cfg["wrt"])

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

# Left: absolute forward and forward+backward times. Right: the ratio, which is
# the claim. Rows are grouped by mode so a jitted point is never joined to an
# eager one.
ax = axes[0]
for bi, basis in enumerate(bases):
    for lane in lanes:
        sel = sorted(
            (r for r in recs if r["basis"] == basis and r["lane"] == lane),
            key=lambda r: r["n"],
        )
        if not sel:
            continue
        ax.plot([r["n"] for r in sel], [r["forward_min_s"] for r in sel],
                marker=style.MARKERS[bi % len(style.MARKERS)], linestyle="-",
                color=style.entity_color("forward"), markersize=3.4,
                label=f"forward - {basis}/{lane}")
        both = [r for r in sel if r["grads"].get("both", {}).get("forward_backward_min_s")]
        if both:
            ax.plot([r["n"] for r in both],
                    [r["grads"]["both"]["forward_backward_min_s"] for r in both],
                    marker=style.MARKERS[bi % len(style.MARKERS)], linestyle="--",
                    color=style.entity_color("forward_backward"), markersize=3.4,
                    label=f"fwd+bwd - {basis}/{lane}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$N$"); ax.set_ylabel("wall-clock [s]")
style.finish(ax, legend_kwargs={"loc": "best", "fontsize": 5.8})

ax = axes[1]
for wi, wrt in enumerate(wrts):
    for bi, basis in enumerate(bases):
        for lane in lanes:
            sel = sorted(
                (r for r in recs
                 if r["basis"] == basis and r["lane"] == lane
                 and r["grads"].get(wrt, {}).get("ratio")),
                key=lambda r: r["n"],
            )
            if not sel:
                continue
            ax.plot(
                [r["n"] for r in sel],
                [r["grads"][wrt]["ratio"] for r in sel],
                marker=style.MARKERS[bi % len(style.MARKERS)],
                linestyle=["-", "--", ":"][wi % 3],
                color=style.CATEGORICAL[wi % len(style.CATEGORICAL)],
                label=f"d/d {wrt} - {basis}/{lane}",
                markersize=3.4,
            )
ax.axhline(1.0, color=style.INK_MUTED, linewidth=0.8, zorder=1)
ax.set_xscale("log")
ax.set_xlabel("$N$")
ax.set_ylabel("(forward + backward) / forward")
style.finish(ax, legend_kwargs={"loc": "best", "fontsize": 5.8})

modes = sorted({r["mode"] for r in recs})
axes[1].text(0.02, 0.98, "timed mode: " + ", ".join(modes),
             transform=axes[1].transAxes, va="top", ha="left",
             fontsize=6.5, color=style.INK_MUTED)
fig.tight_layout()
style.footer(
    fig,
    jsonio.config_caption(cfg, ["order", "theta", "leaf_size", "precision", "device"]),
)
style.save(fig, FIG_DIR / "fig12_autodiff_overhead.pdf")

for r in recs:
    for wrt, g in r["grads"].items():
        if g.get("ratio"):
            print(f"N={r['n']:<8d} {r['basis']:<8s} {r['lane']:<12s} {r['mode']:<5s} "
                  f"d/d{wrt:<10s} ratio={g['ratio']:.2f}")


## Caption

Cost of the reverse pass relative to the forward pass for the fixed-topology
differentiable FMM force, against $N$. **Left:** absolute forward and
forward-plus-backward wall-clock. **Right:** their ratio, which is the quantity
that matters -- the translation cascade is linear, so its reverse pass is a
transpose and a small bounded factor is the expected result. Tree construction is
outside both arms; including it in only the forward would flatter the ratio. Each
measurement records whether the call was jitted or ran eagerly (eager dispatch
re-traces per call), and jitted and eager points are never joined. Values from
`results/differentiability/autodiff_overhead.json`.
